# Pipeline de PLN: del texto crudo al analisis

Pipeline completo paso a paso: limpieza, tokenizacion, stopwords, stemming, lematizacion, frecuencia y generacion de texto.

## Ejercicio 1: Carga y limpieza del texto

1. Crea un archivo `corpus.txt` con texto sobre PLN.
2. Lee el contenido.
3. Convierte a minusculas y elimina caracteres especiales (conservando vocales acentuadas y la enie).
4. Muestra estadisticas antes y despues de la limpieza.

In [ ]:
import re

texto_corpus = """
El Procesamiento de Lenguaje Natural (PLN) es un campo de la inteligencia artificial (IA)
que se ocupa de la interaccion entre las computadoras y el lenguaje humano. El PLN permite
a las maquinas leer, interpretar y comprender el lenguaje humano de una manera valiosa.
Algunas aplicaciones comunes del PLN incluyen la traduccion automatica, el analisis de
sentimientos, los chatbots y la generacion de texto. En 2023, los modelos de lenguaje
grandes como GPT-4 han avanzado mucho. Tecnicas como Word2Vec y los transformers han
revolucionado la representacion vectorial del lenguaje. La tokenizacion, la lematizacion
y el analisis morfologico son pasos fundamentales en cualquier pipeline de PLN.
"""

with open("corpus.txt", "w", encoding="utf-8") as f:
    f.write(texto_corpus)

with open("corpus.txt", "r", encoding="utf-8") as f:
    texto = f.read()

texto_lower = texto.lower()
# Mantiene letras, vocales acentuadas, enie y espacios
texto_limpio = re.sub(r'[^a-zaeiouaeiou\s]', '', texto_lower)
texto_limpio = re.sub(r'[^a-z\xc1\xe1\xe9\xed\xf3\xfa\xfc\xf1\s]', '', texto_lower)

# Forma mas sencilla con la tabla de caracteres validos
VALIDOS = set('abcdefghijklmnopqrstuvwxyz aeiouaeiouñ \n')
texto_limpio = ''.join(c if c in VALIDOS else ' ' for c in texto_lower)
texto_limpio = re.sub(r'\s+', ' ', texto_limpio).strip()

print("--- Texto original (fragmento) ---")
print(texto.strip()[:200])
print("\n--- Texto limpio (fragmento) ---")
print(texto_limpio[:200])
print(f"\nCaracteres: {len(texto.strip())} -> {len(texto_limpio)}")
print(f"Palabras  : {len(texto.split())} -> {len(texto_limpio.split())}")


## Ejercicio 2: Tokenizacion y eliminacion de stopwords

1. Descarga los recursos de NLTK necesarios.
2. Carga las stopwords en espanol.
3. Tokeniza el texto limpio.
4. Filtra las stopwords y muestra estadisticas detalladas.

In [ ]:
import nltk
from nltk.tokenize import word_tokenize
from nltk.corpus import stopwords
from collections import Counter

for ruta, nombre in [('corpora/stopwords','stopwords'),
                      ('tokenizers/punkt','punkt'),
                      ('tokenizers/punkt_tab','punkt_tab')]:
    try:
        nltk.data.find(ruta)
    except LookupError:
        nltk.download(nombre, quiet=True)

stop_words = set(stopwords.words('spanish'))
print(f"Stopwords en espanol: {len(stop_words)}")

tokens = word_tokenize(texto_limpio, language='spanish')
tokens_filtrados = [t for t in tokens if t not in stop_words]
eliminadas = [t for t in tokens if t in stop_words]

freq_sw = Counter(eliminadas)

print(f"\nTokens originales    : {len(tokens)}")
print(f"Tokens filtrados     : {len(tokens_filtrados)}")
print(f"Stopwords eliminadas : {len(eliminadas)} ({len(eliminadas)/len(tokens)*100:.1f}%)")
print(f"Vocabulario restante : {len(set(tokens_filtrados))} palabras unicas")

print("\nTop 5 stopwords eliminadas del texto:")
for sw, n in freq_sw.most_common(5):
    print(f"  '{sw}': {n} veces")

print("\nPrimeros 15 tokens significativos:")
print(tokens_filtrados[:15])


## Ejercicio 3: Stemming vs Lematizacion

1. Aplica stemming con `SnowballStemmer`.
2. Aplica lematizacion con spaCy.
3. Compara los resultados en una tabla.

In [ ]:
from nltk.stem import SnowballStemmer

stemmer = SnowballStemmer('spanish')
tokens_stem = [stemmer.stem(p) for p in tokens_filtrados]

try:
    import spacy
    nlp = spacy.load('es_core_news_sm')
    doc = nlp(' '.join(tokens_filtrados))
    tokens_lema = [t.lemma_ for t in doc]
    print("Lematizacion con spaCy disponible")
except (ImportError, OSError):
    tokens_lema = tokens_filtrados
    print("spaCy no disponible, usando tokens originales como lemas")

print(f"\n{'Original':<22} {'Stemming':<22} {'Lema'}")
print("-" * 66)
for orig, stem, lema in zip(tokens_filtrados[:12], tokens_stem[:12], tokens_lema[:12]):
    print(f"  {orig:<20} {stem:<20} {lema}")

print(f"\nVocabulario unico:")
print(f"  Original : {len(set(tokens_filtrados))}")
print(f"  Stemming : {len(set(tokens_stem))}")
print(f"  Lemas    : {len(set(tokens_lema))}")


## Ejercicio 4: Frecuencia y visualizacion

1. Calcula la distribucion de frecuencia de los lemas.
2. Obtiene el top 10.
3. Crea un grafico de barras.

In [ ]:
from nltk.probability import FreqDist
import matplotlib.pyplot as plt

fdist = FreqDist(tokens_lema)
top10 = fdist.most_common(10)

print("Top 10 palabras mas frecuentes (lemas):")
for i, (p, n) in enumerate(top10, 1):
    barra = '#' * n
    print(f"  {i:2}. {p:<20} {barra} ({n})")

palabras_top = [p[0] for p in top10]
frecs_top    = [p[1] for p in top10]

fig, ax = plt.subplots(figsize=(10, 5))
ax.bar(palabras_top, frecs_top, color='steelblue', edgecolor='black', linewidth=0.5)
for i, (p, n) in enumerate(zip(palabras_top, frecs_top)):
    ax.text(i, n + 0.05, str(n), ha='center', fontsize=9, fontweight='bold')
ax.set_title('Top 10 Palabras mas Frecuentes (Lemas)')
ax.set_xlabel('Palabras')
ax.set_ylabel('Frecuencia')
ax.set_ylim(0, max(frecs_top) + 1)
plt.xticks(rotation=35, ha='right')
plt.tight_layout()
plt.show()


## Ejercicio 5: Generacion de texto con modelo de Markov

1. Construye un modelo de Markov de orden 1 y orden 2.
2. Genera oraciones con ambos modelos.
3. Compara la coherencia del texto generado.

In [ ]:
import random

def construir_markov(tokens, orden=1):
    modelo = {}
    for i in range(len(tokens) - orden):
        clave = tuple(tokens[i:i+orden])
        sig   = tokens[i+orden]
        modelo.setdefault(clave, []).append(sig)
    return modelo

def generar(modelo, longitud=8, orden=1):
    clave = random.choice(list(modelo.keys()))
    resultado = list(clave)
    for _ in range(longitud - orden):
        sig = modelo.get(tuple(resultado[-orden:]))
        if not sig:
            break
        resultado.append(random.choice(sig))
    return ' '.join(resultado)

random.seed(42)
mk1 = construir_markov(tokens_lema, orden=1)
mk2 = construir_markov(tokens_lema, orden=2)

print("Orden 1 (menos coherente):")
for _ in range(3):
    print(f"  {generar(mk1, longitud=9, orden=1)}")

print("\nOrden 2 (mas coherente):")
for _ in range(3):
    print(f"  {generar(mk2, longitud=9, orden=2)}")
